# Middleware

Middleware provides a way to more tightly control what happens inside the agent . Middleware is useful for the following:

* Tracking agent behavior with logging , analytics and debugging
* Transforming prompts , tool selection and output formatting 
* Adding retries , fallbacks and early termination logic 
* applying rate limits , guardrails and PII detection 


In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage , SystemMessage

### MessageBased summarization 

agent= create_agent(
    model= "groq:qwen/qwen3-32b",
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:qwen/qwen3-32b",
            trigger =("messages",10),
            keep=("messages",4)
        )
    ]
)




In [9]:
### Run with thread id
config = {"configurable":{"thread_id":"test-1"}}


In [10]:
questions = [
    "what is 2+2",
    "what is 10*5",
    "what is 100/4",
    "what is 15-7",
    "what is 3*3",
    "what is 4*4",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"messages: {response}")
    print(f"Messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='106a6761-dc72-4167-a43b-5d0f0429308f'), AIMessage(content='<think>\nOkay, so I need to figure out what 2 + 2 is. Let\'s start from the basics. Addition is one of the fundamental operations in arithmetic, right? When you add two numbers together, you\'re combining their quantities. So, 2 and 2 are both whole numbers. Let me visualize this. If I have two apples and then I get two more apples, how many apples do I have in total? That should be four apples. But maybe I should break it down further.\n\nLet me count. Starting with 2, if I add 1, that\'s 3, and then add another 1, that makes 4. So, 2 + 1 + 1 equals 4. Alternatively, since addition is associative and commutative, the order doesn\'t matter. So 2 + 2 is the same as 2 + (1 + 1), which is (2 + 1) + 1, which we already saw gives 4. \n\nAnother way to think about it is using the number line. If I start at 2 and move 2 units to

# Token size

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage , SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city:str)->str:
    """search hotels - returns long response to use more tokens. """
    return f"""Hotels in {city}:
    1.Grand hotell - 5 star , $200/night, spa , pool ,gym
    2.City Inn - 4 star , $100/night , business center
    3.Budget stay - 3 star , $50/night , free wifi"""

agent = create_agent(
    model ="groq:qwen/qwen3-32b",
    tools =[search_hotels],
    checkpointer= InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model= "groq:qwen/qwen3-32b",
            trigger=("tokens",500),
            keep= ("tokens",200),
             )
    ]
)

config = {"configurable" : {"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars = sum(len(str(m.content))for m in messages)
    return total_chars // 4  #4 characters = 1 token





In [29]:
# run test

cities = ["Paris","London","Tokyo","New york","Dubai","singapore"]

for city in cities :
    response = agent.invoke (
        {"messages" : [HumanMessage(content = f"Find hotels in {city}")]},
        config=config 

    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens,{len(response['messages'])}messsages")
    print(f"{(response['messages'])}")

Paris: ~148 tokens,4messsages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e7ec04c7-cae9-4677-a0ec-761388247a25'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to find hotels in Paris. Let me check the tools provided. There's a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens. Since the user's query is straightforward, I should call this function with the city set to Paris. I need to make sure the arguments are correctly formatted as JSON within the tool_call tags. Let me structure the response properly.\n", 'tool_calls': [{'id': 'zdm1xvv1x', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 155, 'total_tokens': 270, 'completion_time': 0.181389195, 'completion_tokens_details': {'reasoning_tokens': 90}, 

KeyboardInterrupt: 

# Fraction

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # groq:qwen/qwen3-32b context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~63 tokens (0.0492%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='e62563fc-502a-41dc-b309-33ce3a2a781b'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the tools available. There's a function called search_hotels that requires a city parameter. Since the user mentioned Paris, I need to call that function with the city set to Paris. I should make sure to format the tool call correctly within the XML tags. Let me structure the JSON with the name of the function and the arguments as specified. Alright, that should do it.\n", 'tool_calls': [{'id': '4evxb8jcw', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 147, 'total_tokens': 262, 'completion_time': 0.177819254, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0